# IUCNAssessmentReviewer Test

This notebook tests the current `AssessmentParser` and `IUCNAssessmentReviewer` directly.
By default it uses the bundled sample JSON, but you can point it at your own assessment JSON by editing the path cell below.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

WORKDIR = Path.cwd().resolve()
REPO_ROOT = None
for candidate in [WORKDIR, *WORKDIR.parents]:
    if (candidate / "iucn_rules_checker").exists():
        REPO_ROOT = candidate
        break
    if candidate.name == "iucn_rules_checker":
        REPO_ROOT = candidate.parent
        break

assert REPO_ROOT is not None, f"Could not find repository root from {WORKDIR}"
PACKAGE_ROOT = REPO_ROOT / "iucn_rules_checker"
assert PACKAGE_ROOT.exists(), f"Could not find package folder from {WORKDIR}"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def fresh_project_imports(verbose: bool = True):
    importlib.invalidate_caches()
    stale_modules = [
        name for name in list(sys.modules)
        if name == "iucn_rules_checker" or name.startswith("iucn_rules_checker.")
    ]
    for module_name in sorted(stale_modules, key=len, reverse=True):
        del sys.modules[module_name]

    assessment_parser_module = importlib.import_module("iucn_rules_checker.assessment_parser")
    checkers_package = importlib.import_module("iucn_rules_checker.checkers")
    base_module = importlib.import_module("iucn_rules_checker.checkers.base")
    assessment_reviewer_module = importlib.import_module("iucn_rules_checker.assessment_reviewer")

    checker_package_locations = [
        Path(path).resolve()
        for path in (checkers_package.__spec__.submodule_search_locations or [])
    ]

    result = {
        "stale_module_count": len(stale_modules),
        "assessment_parser_module": assessment_parser_module,
        "base_module": base_module,
        "assessment_reviewer_module": assessment_reviewer_module,
        "checker_package_locations": checker_package_locations,
        "AssessmentParser": assessment_parser_module.AssessmentParser,
        "IUCNAssessmentReviewer": assessment_reviewer_module.IUCNAssessmentReviewer,
    }

    if verbose:
        print(f"Cleared {result['stale_module_count']} in-memory iucn_rules_checker modules before import.")
        print("Using parser from:", Path(assessment_parser_module.__file__).resolve())
        print("Using checker package search locations:", checker_package_locations)
        print("Using base checker from:", Path(base_module.__file__).resolve())
        print("Using reviewer from:", Path(assessment_reviewer_module.__file__).resolve())

    return result


project_modules = fresh_project_imports(verbose=True)
AssessmentParser = project_modules["AssessmentParser"]
IUCNAssessmentReviewer = project_modules["IUCNAssessmentReviewer"]



In [ ]:
# Set CUSTOM_JSON_PATH to a string or Path for your own assessment JSON.
# Leave it as None to use the bundled sample file.
CUSTOM_JSON_PATH = None

DEFAULT_JSON_PATH = PACKAGE_ROOT / "test_json_file" / "Acianthera odontotepala_draft_status_Jun2025 (1).json"
JSON_PATH = Path(CUSTOM_JSON_PATH).expanduser() if CUSTOM_JSON_PATH else DEFAULT_JSON_PATH
assert JSON_PATH.exists(), f"Could not find {JSON_PATH}"
print(JSON_PATH)


In [ ]:
with JSON_PATH.open(encoding="utf-8") as handle:
    assessment = json.load(handle)

print(type(assessment).__name__)
print("Top-level keys:", list(assessment.keys()))
print("Title:", assessment["title"])
print("Block count:", len(assessment["blocks"]))
print("Child count:", len(assessment["children"]))

dict
Top-level keys: ['title', 'level', 'path', 'blocks', 'children', 'comments']
Title: Acrocarpus_fraxinifolius_JP (2)
Block count: 6
Child count: 9


## Run the parser

The parser should return a `dict[str, str]` where each key is a full block path and each value is the text for that individual block. Paragraph blocks stay as single entries, while table blocks are now split into individual row entries using keys like `[table 1] [row 1]` and `[table 1] [row 2]`.

The parser now uses `text_rich` and `rows_rich` only. It ignores plain `text` / `rows` fallback content, and it also ignores `style` blocks entirely. In the current JSON, any rich-text markup in the parsed output now comes directly from the stored rich fields themselves.


In [ ]:
project_modules = fresh_project_imports(verbose=False)
parser = project_modules["AssessmentParser"]()
full_report = parser.parse(assessment)

paragraph_entries = [key for key in full_report if "[paragraph " in key]
table_row_entries = [key for key in full_report if "[table " in key and "[row " in key]

print(type(full_report).__name__)
print("Parsed entries:", len(full_report))
print("Paragraph blocks:", len(paragraph_entries))
print("Table rows:", len(table_row_entries))
print()

for index, section_name in enumerate(full_report.keys(), start=1):
    print(f"{index:>2}. {section_name}")
    if index == 10:
        break



dict
Parsed entries: 79
Paragraph blocks: 33
Table rows: 46

 1. Acrocarpus_fraxinifolius_JP (2) [paragraph 1]
 2. Acrocarpus_fraxinifolius_JP (2) [paragraph 2]
 3. Acrocarpus_fraxinifolius_JP (2) [paragraph 3]
 4. Acrocarpus_fraxinifolius_JP (2) [paragraph 4]
 5. Acrocarpus_fraxinifolius_JP (2) [table 1] [row 1]
 6. Acrocarpus_fraxinifolius_JP (2) [table 1] [row 2]
 7. Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment Information [paragraph 1]
 8. Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment Information [paragraph 2]
 9. Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment Rationale [paragraph 1]
10. Acrocarpus_fraxinifolius_JP (2) > Distribution > Geographic Range [paragraph 1]


## Full report

This cell prints the complete `full_report` mapping returned by `AssessmentParser.parse(...)`.

In [ ]:
print(json.dumps(full_report, indent=2, ensure_ascii=False))

{
  "Acrocarpus_fraxinifolius_JP (2) [paragraph 1]": "<b>Draft</b>",
  "Acrocarpus_fraxinifolius_JP (2) [paragraph 2]": "<b><i>Acrocarpus fraxinifolius</i></b> <b>- Arn.</b>",
  "Acrocarpus_fraxinifolius_JP (2) [paragraph 3]": "PLANTAE - TRACHEOPHYTA - MAGNOLIOPSIDA - FABALES - FABACEAE - Acrocarpus - fraxinifolius",
  "Acrocarpus_fraxinifolius_JP (2) [paragraph 4]": "<b>Common Names:</b> Mundane (English), Australian ash, Indian ash, Kenya coffeeshade (English), Mandania, Mndanii (Sino-Tibetan (Other)), Pink Cedar (English), Shingle Tree (English)\n<b>Synonyms:</b> <b><i>Acrocarpus fraxinifolius</i></b> var. guangxiensis S.L.Mo & Y.Wei; Acrocarpus grandis (Miq.) Miq.; Acrocarpus combretiflorus Teijsm. & Binn.; Mezoneurum grande Miq.",
  "Acrocarpus_fraxinifolius_JP (2) [table 1] [row 1]": "Red List Status",
  "Acrocarpus_fraxinifolius_JP (2) [table 1] [row 2]": "LC - Least Concern, (IUCN version 3.1)",
  "Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment Information [

In [ ]:
def preview_section(report: dict[str, str], section_hint: str, char_limit: int = 400) -> None:
    matches = [key for key in report if section_hint.lower() in key.lower()]
    if not matches:
        print(f"No section matched: {section_hint}")
        return

    section_name = matches[0]
    text = report[section_name]
    print(section_name)
    print("-" * len(section_name))
    print(text[:char_limit] + ("..." if len(text) > char_limit else ""))

In [ ]:
preview_section(full_report, "Assessment Information")
print()
preview_section(full_report, "Assessment Rationale")
print()
preview_section(full_report, "Bibliography")

Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment Information [paragraph 1]
--------------------------------------------------------------------------------------------
<b>Date of Assessment:</b> 2022-08-01

Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment Rationale [paragraph 1]
------------------------------------------------------------------------------------------
<i>Acrocarpus fraxinifolius</i> is a widespread species distributed throughout tropical Asia and Africa and Central America, where its natural and biological distribution covers southeastern Asia. This species is widely planted both within and outside its natural distribution and has become naturalized in several countries. The population information is limited for this species, but the current po...

Acrocarpus_fraxinifolius_JP (2) > Bibliography [paragraph 1]
------------------------------------------------------------
Cheng, W.J. 1985. <i>Tree flora of china. Vol II</i>. China Forest

## Sanity checks

These assertions document the parser's current behavior. The parser now collects paragraph blocks and individual table rows as separate entries, avoids merged section-level keys when a section contains multiple blocks, and prefers the rich block fields when they are available.


In [ ]:
title = assessment["title"]
expected_entries = [
    f"{title} [paragraph 1]",
    f"{title} [paragraph 2]",
    f"{title} [table 1] [row 1]",
    f"{title} [table 1] [row 2]",
    f"{title} > Red List Assessment > Assessment Information [paragraph 1]",
    f"{title} > Red List Assessment > Assessment Rationale [paragraph 1]",
    f"{title} > Bibliography [paragraph 1]",
]

bibliography_entries = [key for key in full_report if " > Bibliography [" in key]
rationale_1 = f"{title} > Red List Assessment > Assessment Rationale [paragraph 1]"

assert isinstance(full_report, dict)
assert len(full_report) == 79
assert len(paragraph_entries) == 33
assert len(table_row_entries) == 46
assert all(isinstance(key, str) for key in full_report)
assert all(isinstance(value, str) and value.strip() for value in full_report.values())
assert not any("[style " in key for key in full_report)

for entry in expected_entries:
    assert entry in full_report

assert len(bibliography_entries) >= 1
assert full_report[expected_entries[0]] == "Draft"
assert full_report[expected_entries[1]] == "Acrocarpus fraxinifolius - Arn."
assert full_report[expected_entries[2]] == "Red List Status"
assert "LC - Least Concern" in full_report[expected_entries[3]]
assert "Date of Assessment: 2022-08-01" in full_report[expected_entries[4]]
assert "Acrocarpus fraxinifolius is a widespread species" in full_report[expected_entries[5]]
assert "Tree flora of china. Vol II" in full_report[expected_entries[6]]
assert "<sup>2</sup>" in full_report[rationale_1]
assert not any("&amp;" in value for value in full_report.values())
assert not any("-&gt;" in value for value in full_report.values())

rich_demo = {
    "title": "Root",
    "blocks": [
        {"type": "paragraph", "text": "Area 10 km2", "text_rich": "Area 10 km<sup>2</sup>"},
        {"type": "table", "rows": [["CO2"]], "rows_rich": [["CO<sub>2</sub>"]]},
        {"type": "paragraph", "text": "Plain only paragraph"},
        {"type": "table", "rows": [["Plain only row"]]},
        {"type": "style", "data": {"bold": ["Area"], "italic": ["Area"]}},
    ],
    "children": [],
}
rich_report = parser.parse(rich_demo)
assert rich_report["Root [paragraph 1]"] == "Area 10 km<sup>2</sup>"
assert rich_report["Root [table 1] [row 1]"] == "CO<sub>2</sub>"
assert "Root [paragraph 2]" not in rich_report
assert "Root [table 2] [row 1]" not in rich_report

print("AssessmentParser sanity checks passed.")


AssessmentParser sanity checks passed.


In [ ]:
block_lengths = sorted(
    ((section_name, len(text)) for section_name, text in full_report.items()),
    key=lambda item: item[1],
    reverse=True,
)

for section_name, length in block_lengths[:10]:
    print(f"{length:>5} chars | {section_name}")


 1138 chars | Acrocarpus_fraxinifolius_JP (2) > Habitats and Ecology [paragraph 1]
 1060 chars | Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment Rationale [paragraph 1]
  718 chars | Acrocarpus_fraxinifolius_JP (2) > Use and Trade > General Use and Trade Information [paragraph 1]
  538 chars | Acrocarpus_fraxinifolius_JP (2) > Distribution > Geographic Range [paragraph 1]
  406 chars | Acrocarpus_fraxinifolius_JP (2) > Threats [paragraph 1]
  371 chars | Acrocarpus_fraxinifolius_JP (2) [paragraph 4]
  354 chars | Acrocarpus_fraxinifolius_JP (2) > Distribution > Map Status [table 1] [row 2]
  295 chars | Acrocarpus_fraxinifolius_JP (2) > Population [paragraph 1]
  252 chars | Acrocarpus_fraxinifolius_JP (2) > Distribution > Map Status [table 1] [row 1]
  224 chars | Acrocarpus_fraxinifolius_JP (2) > Bibliography [paragraph 12]


## Run the reviewer

The reviewer consumes the parsed `full_report` mapping and applies the currently enabled checker classes to each `(section_name, section_text)` pair. In the current configuration, `SymbolChecker` and `LanguageChecker` are excluded.

Because the parser now prefers the rich block fields and splits tables into per-row entries, the reviewer sees superscript and subscript markup from the source where present and can inspect table rows individually instead of as one merged table string.


In [ ]:
from collections import Counter

project_modules = fresh_project_imports(verbose=False)
reviewer = project_modules["IUCNAssessmentReviewer"]()
violations = reviewer.review_full_report(full_report)
configured_checkers = [type(checker).__name__ for checker in reviewer.checkers]
rule_class_counts = dict(sorted(Counter(violation.rule_class for violation in violations).items()))

print(type(violations).__name__)
print("Configured checkers:", configured_checkers)
print("Violation count:", len(violations))
print("Rule-class counts:", rule_class_counts)



list
Configured checkers: ['AbbreviationChecker', 'BibliographyChecker', 'DateChecker', 'FormattingChecker', 'GeographyChecker', 'IUCNTermsChecker', 'NumberChecker', 'PunctuationChecker', 'ReferenceChecker', 'ScientificNameChecker', 'SpellingChecker']
Violation count: 71
Rule-class counts: {'AbbreviationChecker': 4, 'BibliographyChecker': 6, 'FormattingChecker': 9, 'GeographyChecker': 3, 'NumberChecker': 32, 'PunctuationChecker': 15, 'ReferenceChecker': 1, 'SpellingChecker': 1}


## Full violation list

This cell prints the full list returned by `IUCNAssessmentReviewer.review_full_report(...)` on the parsed assessment report.


In [ ]:
print(json.dumps([violation.to_dict() for violation in violations], indent=2, ensure_ascii=False))


[
  {
    "rule_class": "FormattingChecker",
    "rule_method": "FormattingChecker.check_genus_and_species",
    "section_name": "Acrocarpus_fraxinifolius_JP (2)",
    "matched_text": "Acrocarpus",
    "matched_snippet": "& Y.Wei; Acrocarpus grandis (Miq.)",
    "message": "Scientific names should be italicized and use correct case: '<i>Acrocarpus</i>'",
    "suggested_fix": "<i>Acrocarpus</i>"
  },
  {
    "rule_class": "FormattingChecker",
    "rule_method": "FormattingChecker.check_genus_and_species",
    "section_name": "Acrocarpus_fraxinifolius_JP (2)",
    "matched_text": "Acrocarpus",
    "matched_snippet": "(Miq.) Miq.; Acrocarpus combretiflorus Teijsm.",
    "message": "Scientific names should be italicized and use correct case: '<i>Acrocarpus</i>'",
    "suggested_fix": "<i>Acrocarpus</i>"
  },
  {
    "rule_class": "NumberChecker",
    "rule_method": "NumberChecker.check_large_numbers",
    "section_name": "Acrocarpus_fraxinifolius_JP (2) > Red List Assessment > Assessment R

## Reviewer sanity checks

These assertions document the reviewer's current wiring and confirm that multiple checker categories are active on the sample assessment.


In [ ]:
expected_checkers = [
    "AbbreviationChecker",
    "BibliographyChecker",
    "DateChecker",
    "FormattingChecker",
    "GeographyChecker",
    "IUCNTermsChecker",
    "NumberChecker",
    "PunctuationChecker",
    "ReferenceChecker",
    "ScientificNameChecker",
    "SpellingChecker",
]

assert isinstance(violations, list)
assert all(hasattr(violation, "rule_class") for violation in violations)
assert all(hasattr(violation, "message") for violation in violations)
assert all(hasattr(violation, "section_name") for violation in violations)
assert configured_checkers == expected_checkers
assert "SymbolChecker" not in configured_checkers
assert "LanguageChecker" not in configured_checkers
assert len(violations) > 0
assert len(rule_class_counts) >= 5
for rule_class in [
    "AbbreviationChecker",
    "BibliographyChecker",
    "NumberChecker",
    "PunctuationChecker",
    "ReferenceChecker",
]:
    assert rule_class in rule_class_counts

print("IUCNAssessmentReviewer sanity checks passed.")


IUCNAssessmentReviewer sanity checks passed.
